# California Yoga Studio Reviews
## Notebook 1: Metadata Preparation
*2026-05*

This notebook downloads the UCSD Google Local California metadata file,
filters it to yoga-related businesses, removes duplicates and non-yoga
businesses through iterative category inspection, and saves the cleaned
studio metadata for use in subsequent notebooks.

**Dataset source:** [UCSD Google Local Dataset](https://mcauleylab.ucsd.edu/public_datasets/gdrive/googlelocal/)
**Output:** `yoga_studios_meta.pkl` — 423 yoga studios across California

## Setup

In [ ]:
# imports
from google.colab import drive
import os
import urllib.request
import gzip
import json
import pandas as pd
from collections import Counter
import pickle

In [ ]:
# seting-up the google drive
drive.mount('/content/gdrive', force_remount=True)

PROJECT_DIR = '/content/gdrive/MyDrive/YogaStudioReviews'
os.makedirs(PROJECT_DIR, exist_ok=True)

Mounted at /content/gdrive


In [ ]:
PROJECT_DIR = '/content/gdrive/MyDrive/YogaStudioReviews'
META_URL = 'https://mcauleylab.ucsd.edu/public_datasets/gdrive/googlelocal/meta-California.json.gz'
META_PATH = os.path.join(PROJECT_DIR, 'meta-California.json.gz')

## Download metadata

In [ ]:
# downloading the metadata file
META_URL = 'https://mcauleylab.ucsd.edu/public_datasets/gdrive/googlelocal/meta-California.json.gz'
META_PATH = os.path.join(PROJECT_DIR, 'meta-California.json.gz')

print('Downloading metadata...')
urllib.request.urlretrieve(META_URL, META_PATH)
print('Done.')

Done.


## Parse and inspect raw data

In [ ]:
# Defining the parser function
def parse(path):
    with gzip.open(path, 'rt', encoding='utf-8') as f:
        for line in f:
            try:
                yield json.loads(line)
            except json.JSONDecodeError:
                continue

In [ ]:
# Viewing the first 3 records
print('Sample raw metadata records:\n')
for i, record in enumerate(parse(META_PATH)):
    print(f'--- Record {i+1} ---')
    print(json.dumps(record, indent=2))
    print()
    if i >= 2:
        break

Sample raw metadata records:

--- Record 1 ---
{
  "name": "City Textile",
  "address": "City Textile, 3001 E Pico Blvd, Los Angeles, CA 90023",
  "gmap_id": "0x80c2c98c0e3c16fd:0x29ec8a728764fdf9",
  "description": null,
  "latitude": 34.0188913,
  "longitude": -118.2152898,
  "category": [
    "Textile exporter"
  ],
  "avg_rating": 4.5,
  "num_of_reviews": 6,
  "price": null,
  "hours": null,
  "MISC": null,
  "state": "Open now",
  "relative_results": [
    "0x80c2c624136ea88b:0xb0315367ed448771",
    "0x80c2c97cb7c52f17:0xb66ee68c1c366f2d"
  ],
  "url": "https://www.google.com/maps/place//data=!4m2!3m1!1s0x80c2c98c0e3c16fd:0x29ec8a728764fdf9?authuser=-1&hl=en&gl=us"
}

--- Record 2 ---
{
  "name": "San Soo Dang",
  "address": "San Soo Dang, 761 S Vermont Ave, Los Angeles, CA 90005",
  "gmap_id": "0x80c2c778e3b73d33:0xbdc58662a4a97d49",
  "description": null,
  "latitude": 34.0580917,
  "longitude": -118.2921295,
  "category": [
    "Korean restaurant"
  ],
  "avg_rating": 4.4,
  "

## Category inspection

In [ ]:
# Inspecting all unique category values across all records
print('Scanning category values...')

category_counts = {}

for record in parse(META_PATH):
    categories = record.get('category') or []
    for cat in categories:
        category_counts[cat] = category_counts.get(cat, 0) + 1

print(f'\nTotal unique categories in California: {len(category_counts):,}')

# Sort by frequency
sorted_categories = sorted(category_counts.items(), key=lambda x: x[1], reverse=True)

# Showing all yoga-related ones if there are any
print('\nAll categories containing "yoga":')
for cat, count in sorted_categories:
    if 'yoga' in cat.lower():
        print(f'  {count}  {cat}')

Scanning category values...

Total unique categories in California: 4,019

All categories containing "yoga":
  951  Yoga studio
  52  Yoga instructor
  12  Yoga retreat center
  8  Bikram yoga studio


### Category filtering decisions

After inspecting all yoga-related categories, three core categories were
selected for inclusion:

- **Yoga studio (951)** — core dataset
- **Yoga instructor (52)** — individual instructors; kept as plausibly
  relevant but may behave differently from studios
- **Bikram yoga studio (8)** — a specific yoga style; kept

**Yoga retreat center (12)** was excluded — retreat reviews
discuss accommodation, food, and scenery rather than studio experience,
which would pollute the NLP analysis.

## Filter to yoga studios

Two-stage filtering is applied:
1. Businesses must have at least one core yoga category
2. Businesses must not have any clearly non-yoga category (general gyms,
   sports facilities, unrelated businesses that were miscategorised)

The exclusion list was built by inspecting all unique categories
found across the initial yoga business set.

In [ ]:
# Defining the categories to keep
YOGA_CATEGORIES = {
    'Yoga studio',
    'Yoga instructor',
    'Bikram yoga studio'
}

# Filtering to yoga businesses
print('Filtering to yoga studios...')
yoga_records = []

for record in parse(META_PATH):
    categories = record.get('category') or []
    has_yoga = any(cat in YOGA_CATEGORIES for cat in categories)
    has_retreat = 'Yoga retreat center' in categories

    if has_yoga and not has_retreat:
        yoga_records.append(record)

yoga_meta = pd.DataFrame(yoga_records)
print(f'Found {len(yoga_meta)} yoga businesses in California')
print(f'Breakdown by category:')

for cat in YOGA_CATEGORIES:
    count = yoga_meta['category'].apply(
        lambda cats: cat in (cats or [])
    ).sum()
    print(f'  {count:>5}  {cat}')

Filtering to yoga studios...
Found 984 yoga businesses in California
Breakdown by category:
      8  Bikram yoga studio
     50  Yoga instructor
    950  Yoga studio


In [ ]:
# Checking for duplicate business IDs
duplicate_count = yoga_meta['gmap_id'].duplicated().sum()
print(f'Duplicate count: {duplicate_count}')

Duplicate count: 9


In [ ]:
# Filtering the duplicates and keeping the first occurrence
if duplicate_count > 0:
    yoga_meta = yoga_meta.drop_duplicates(subset='gmap_id', keep='first')
    print(f'After duplicate removal: {len(yoga_meta)} studios')

After duplicate removal: 975 studios


Short exploration to see if everything looks decent.

In [ ]:
# How many studios?
print(f'Total unique yoga businesses: {len(yoga_meta)}')
print(f'\nColumns available:')
print(yoga_meta.columns.tolist())

Total unique yoga businesses: 975

Columns available:
['name', 'address', 'gmap_id', 'description', 'latitude', 'longitude', 'category', 'avg_rating', 'num_of_reviews', 'price', 'hours', 'MISC', 'state', 'relative_results', 'url']


In [ ]:
# Category breakdown
print('Category breakdown:')
for cat in YOGA_CATEGORIES:
    count = yoga_meta['category'].apply(
        lambda cats: cat in (cats or [])
    ).sum()
    print(f'  {count}  {cat}')

Category breakdown:
  8  Bikram yoga studio
  50  Yoga instructor
  941  Yoga studio


In [ ]:
# Get all unique categories across all 987 yoga businesses
all_categories = Counter()
for cats in yoga_meta['category']:
    for cat in (cats or []):
        all_categories[cat] += 1

print(f'Total unique categories found: {len(all_categories)}')
print('\nAll categories sorted by frequency:')
for cat, count in all_categories.most_common():
    print(f'  {count}  {cat}')

Total unique categories found: 235

All categories sorted by frequency:
  941  Yoga studio
  472  Gym
  419  Fitness center
  303  Personal trainer
  143  Physical fitness program
  95  Pilates studio
  52  Massage therapist
  52  Weightlifting area
  50  Yoga instructor
  41  Weight loss service
  39  Meditation center
  35  Swimming pool
  34  Rock climbing gym
  33  Spa
  33  Boot camp
  33  Aerobics instructor
  32  Training centre
  31  Wellness center
  30  Indoor cycling
  29  Non-profit organization
  25  Community center
  24  Summer camp
  21  Meditation instructor
  20  Boutique
  18  Wellness program
  18  Martial arts school
  17  Boxing gym
  16  Swim club
  14  Holistic medicine practitioner
  14  Racquetball club
  13  Sauna
  13  Event venue
  12  Spa and health club
  12  Youth social services organization
  12  Rock climbing
  11  Recreation center
  11  Reiki therapist
  11  Indoor swimming pool
  10  Dance school
  10  Basketball court
  10  Gymnastics center
  10 

In [ ]:
# Core yoga categories to keep - must have at least one of these
CORE_YOGA_CATEGORIES = {
    'Yoga studio',
    'Yoga instructor',
    'Bikram yoga studio'
}

# Additionally exclude if they have these clearly non-yoga categories
# AND no strong yoga signal
EXCLUDE_CATEGORIES = {
    'Gym',
    'Fitness center',
    'Weightlifting area',
    'Rock climbing gym',
    'Rock climbing',
    'Boxing gym',
    'Muay Thai boxing gym',
    'Racquetball club',
    'Tennis club',
    'Tennis court',
    'Basketball court',
    'Basketball club',
    'Indoor golf course',
    'Squash club',
    'Soccer club',
    'Boot camp',
    'Aerobics instructor',
    'Indoor cycling',
    'Swim club',
    'Swimming pool',
    'Dance school',
    'Martial arts school',
    'Jujitsu school',
    'Judo club',
    'Aikido school',
    'Karate school',
    'Self defense school',
    'Dance company',
    'Adult DVD store',
    'Department of motor vehicles',
    'Farm',
    'Pony ride service',
    "Children's party service",
    'Retreat center',
    'RV park',
    'Campground',
    'Mobile home park',
    'Vacation home rental agency',
    'Winery',
    'Wine bar',
    'Hair salon',
    'Waxing hair removal service',
    'Museum',
    'Bicycle Shop',
    'Bicycle club',
    'Musical club',
    'Garden center',
    'Comedy club'
}

before = len(yoga_meta)

yoga_meta = yoga_meta[
    yoga_meta['category'].apply(
        lambda cats: (
            # Must have at least one core yoga category
            any(cat in CORE_YOGA_CATEGORIES for cat in (cats or [])) and
            # Must not have any clearly non-yoga category
            not any(cat in EXCLUDE_CATEGORIES for cat in (cats or []))
        )
    )
]

print(f'Removed: {before - len(yoga_meta)} businesses')
print(f'Remaining: {len(yoga_meta)} yoga studios')

Removed: 0 businesses
Remaining: 423 yoga studios


In [ ]:
all_categories = Counter()
for cats in yoga_meta['category']:
    for cat in (cats or []):
        all_categories[cat] += 1

print(f'Total unique categories found: {len(all_categories)}')
print('\nAll categories sorted by frequency:')
for cat, count in all_categories.most_common():
    print(f'  {count}  {cat}')

Total unique categories found: 103

All categories sorted by frequency:
  398  Yoga studio
  37  Yoga instructor
  35  Massage therapist
  28  Meditation center
  25  Physical fitness program
  22  Wellness center
  20  Pilates studio
  20  Boutique
  19  Meditation instructor
  11  Holistic medicine practitioner
  10  Wellness program
  10  Reiki therapist
  8  Alternative medicine practitioner
  8  Acupuncture clinic
  8  Chiropractor
  7  Life coach
  7  Bikram yoga studio
  7  Event venue
  6  Tai chi school
  6  Service establishment
  6  Sports massage therapist
  6  Massage spa
  5  Acupuncturist
  5  Physical therapy clinic
  5  Day spa
  5  Skin care clinic
  5  Personal trainer
  5  Health consultant
  4  Nutritionist
  4  Community center
  4  Spa
  4  Clothing store
  3  Astrologer
  3  Buddhist temple
  3  Coworking space
  3  Non-profit organization
  3  Cafe
  2  Family counselor
  2  Hypnotherapy service
  2  Tea house
  2  Thai massage therapist
  2  Family service cen

### Remaining category review

After removing gym and sport-related categories, the remaining 131 unique
categories were reviewed. The vast majority are legitimate yoga and wellness
co-categories (massage therapists, meditation centres, pilates studios,
wellness centres, health food cafés, boutiques). A targeted inspection was
performed on the more ambiguous categories to verify they belonged to
genuine yoga businesses — no further removals were necessary.

## Inspect ambiguous businesses

Businesses with unusual category combinations are inspected manually
to verify they are genuine yoga studios rather than miscategorised entries.

In [ ]:
# Inspect the actual businesses behind the suspicious categories
INSPECT_CATEGORIES = {"Event venue",
  "Day spa",
  "Skin care clinic",
  "Spa",
  "Coworking space",
  "Family counselor",
  "Family service center",
  "Pregnancy care center",
  "Training school",
  "Weight loss service",
  "Medical clinic",
  "Art gallery",
  "Live music venue",
  "Health spa",
  "Professional organizer",
  "Art studio",
  "Women's health clinic",
  "Childbirth class",
  "Special education school",
  "Part time daycare",
  "Chinese medicine clinic",
  "Mental health service",
  "Hiking area",
  "Culinary school",
  "Event planner",
  "Psychic",
  "Outdoor activity organiser",
}

suspicious = yoga_meta[
    yoga_meta['category'].apply(
        lambda cats: any(cat in INSPECT_CATEGORIES for cat in (cats or []))
    )
]

print(f'Businesses to inspect: {len(suspicious)}')
print()
for _, row in suspicious.iterrows():
    print(f"Name: {row['name']}")
    print(f"Categories: {row['category']}")
    print(f"Reviews: {row['num_of_reviews']} | Rating: {row['avg_rating']}")
    print(f"Description: {str(row.get('description', ''))[:100]}")
    print()

Businesses to inspect: 38

Name: iYoga Transformation
Categories: ['Yoga studio', 'Life coach', 'Massage therapist', 'Meditation center', 'Professional organizer']
Reviews: 8 | Rating: 4.7
Description: None

Name: Seventh Chakra Yoga - Placentia
Categories: ['Yoga studio', 'Alternative medicine practitioner', 'Family counselor', 'Life coach', 'Meditation center', 'Meditation instructor', 'Physical fitness program', 'Wellness center', 'Wellness program']
Reviews: 4 | Rating: 4.0
Description: None

Name: Unbound Collective
Categories: ['Wellness center', 'Art studio', 'Meditation center', 'Yoga studio']
Reviews: 2 | Rating: 5.0
Description: None

Name: Aum Vibe
Categories: ['Yoga studio', 'Event venue', 'Tea house']
Reviews: 6 | Rating: 4.8
Description: None

Name: On Point Acupuncture and Wellness Center
Categories: ['Wellness center', 'Acupuncturist', 'Holistic medicine practitioner', 'Massage therapist', 'Meditation instructor', "Women's health clinic", 'Sports massage therapist', 'Th

The data even with the susipicious categories look more or less clean. So no further category sifting will be performed and we stay with 423 yoga studios.

## Dataset exploration

In [ ]:
# How many studios have a description?
has_desc = yoga_meta['description'].notna().sum()
print(f'Studios with description: {has_desc} out of {len(yoga_meta)}')

# Sample a few
print('\nSample descriptions:')
for desc in yoga_meta['description'].dropna().head(5):
    print(f'  — {desc[:120]}...')

Studios with description: 106 out of 423

Sample descriptions:
  — Studio offering Vinyasa, gentle, aerial & prenatal yoga classes, plus Barre workouts....
  — Boutique & studio offering a wide variety of yoga classes, plus meditation, Tai Chi & workshops....
  — Mellow yoga center offering classes & workshops, including therapeutic options & meditation....
  — Vinyasa, yin/yang & restorative yoga classes, plus regular workshops for particular poses....
  — Ocean-view studio offering hot & non-heated Bikram, Ashtanga, Vinyasa & Yin yoga classes....


In [ ]:
# Review counts
print('Review count distribution:')
print(yoga_meta['num_of_reviews'].describe())
print(f'\nStudios with 0 reviews: {(yoga_meta["num_of_reviews"] == 0).sum()}')
print(f'Studios with 1-5 reviews: {((yoga_meta["num_of_reviews"] >= 1) & (yoga_meta["num_of_reviews"] <= 5)).sum()}')
print(f'Studios with 10+ reviews: {(yoga_meta["num_of_reviews"] >= 10).sum()}')

Review count distribution:
count    423.000000
mean      33.241135
std       61.143592
min        1.000000
25%        8.000000
50%       15.000000
75%       28.000000
max      593.000000
Name: num_of_reviews, dtype: float64

Studios with 0 reviews: 0
Studios with 1-5 reviews: 68
Studios with 10+ reviews: 235


In [ ]:
# Rating distributio
print(f'Rating distribution:')
print(yoga_meta['avg_rating'].describe().round(2))

Rating distribution:
count    423.00
mean       4.74
std        0.43
min        1.00
25%        4.70
50%        4.90
75%        5.00
max        5.00
Name: avg_rating, dtype: float64


In [ ]:
# Geographic coverage
print(f'\nStudios with location data:')
print(f'  Latitude available: {yoga_meta["latitude"].notna().sum()}')
print(f'  Longitude available: {yoga_meta["longitude"].notna().sum()}')


Studios with location data:
  Latitude available: 423
  Longitude available: 423


In [ ]:
# Price range distribution if available
if 'price' in yoga_meta.columns:
    print(f'\nPrice range distribution:')
    print(yoga_meta['price'].value_counts(dropna=False))


Price range distribution:
price
None    409
$$       13
$         1
Name: count, dtype: int64


In [ ]:
# Are opening hours available?
print(f'Studios with opening hours: {yoga_meta["hours"].notna().sum()} out of {len(yoga_meta)}')

Studios with opening hours: 349 out of 423


In [ ]:
# Permanently closed studios
if 'state' in yoga_meta.columns:
    closed = yoga_meta['state'].str.contains(
        'permanently closed', case=False, na=False
    ).sum()
    print(f'Permanently closed studios: {closed}')

Permanently closed studios: 63


## Output summary

The filtered dataset contains **423 unique yoga studios** across California:

- All 423 studios have location data (latitude and longitude)
- 896 studios have opening hours data (out of 987 before gym exclusions)
- Mean rating: 4.74 — high but expected given Google Reviews positivity bias
- Mean reviews per studio: 33 — sufficient for corpus-level NLP analysis
- 131 permanently closed studios retained for comparative analysis
- Studios with 0 reviews: none — every studio has at least one data point

The filtered metadata values are saved
for use in the review data preparation notebook.


In [ ]:
# Saving filtered metadata
YOGA_META_PATH = os.path.join(PROJECT_DIR, 'yoga_studios_meta.pkl')
yoga_meta.to_pickle(YOGA_META_PATH)

# Saving gmap_ids for matching to google reviews
yoga_gmap_ids = set(yoga_meta['gmap_id'].tolist())
with open(os.path.join(PROJECT_DIR, 'yoga_gmap_ids.pkl'), 'wb') as f:
    pickle.dump(yoga_gmap_ids, f)